# Expected Returns Analytics
# 
# This notebook analyzes expected returns using the enhanced v2.0 analytics pipeline:
# - **Monte Carlo Simulation** — Probabilistic upside/downside distributions
# - **Price Target Achievement** — Probability-weighted expected returns by sector
# - **Kalman Filtered Targets** — Noise-reduced price target signals
# - **Analyst Sentiment Features** — Feature-level probability analytics
# - **Cross-Model Comparison** — MC vs Kalman vs Achievement model alignment
#
# Data sources: `analytics.monte_carlo_simulation`, `analytics.price_target_achievement`,
# `analytics.kalman_filtered_price_targets`, `analytics.earnings_probability_analysis`


## 1. Setup & Environment Configuration


In [1]:
import warnings
import os
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import plotly.io as pio

warnings.filterwarnings("ignore")

# Configure database connection
if "DB_URL" not in os.environ:
    env_file = "environment_variables.txt"
    if os.path.exists(env_file):
        with open(env_file) as f:
            for line in f:
                line = line.strip()
                if line and not line.startswith("#") and "=" in line:
                    key, value = line.split("=", 1)
                    os.environ[key.strip()] = value.strip()

PLOTLY_TEMPLATE = "plotly_dark"
COLORS = px.colors.qualitative.Dark24

print("✅ Environment configured")

# --- InferenceData schema (ArviZ / xarray bridge) ---
try:
    from finance_ml.analytics.inference_schema import (
        ARVIZ_AVAILABLE,
        build_monte_carlo_inference_data,
        summarize_inference_data,
    )
except ImportError:
    ARVIZ_AVAILABLE = False


✅ Environment configured


## 2. Data Acquisition


In [2]:
%%sql
SELECT * FROM analytics.monte_carlo_simulation

,ticker,name,sector,industry,region,country,exchange,last_price,pt_median,pt_spread,expected_upside_pct,upside_std,var_5_pct,prob_positive_upside,risk_reward_ratio
0,QS,QuantumScape Corporation,Consumer Discretionary,Automobile Components,United States and Canada,US,NasdaqGS,8.82,11.2500,13.500,11.826061,31.582382,-43.962927,65.67,0.374451
1,NWE,NorthWestern Energy Group Inc.,Utilities,Multi-Utilities,United States and Canada,US,NasdaqGS,68.45,64.0000,16.000,-9.348566,4.893435,-18.228299,0.47,-1.910430
2,BIR,Birchcliff Energy Ltd.,Energy,Oil Gas and Consumable Fuels,United States and Canada,CA,TSX,7.20,8.7500,2.000,23.839157,5.685175,14.997369,100.00,4.193214
3,PRCH,Porch Group Inc.,Information Technology,Software,United States and Canada,US,NasdaqCM,7.59,18.0000,9.000,132.641451,24.281265,90.707880,100.00,5.462708
4,CXW,CoreCivic Inc.,Industrials,Commercial Services and Supplies,United States and Canada,US,NYSE,18.50,29.7500,4.000,61.670746,4.428750,54.573432,100.00,13.925090
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2225,ADRO,PT Alamtri Resources Indonesia Tbk,Energy,Oil Gas and Consumable Fuels,Asia / Pacific,ID,IDX,2220.00,2364.1800,1891.344,8.560333,17.529144,-19.960457,67.26,0.488349
2226,CBAV3,Companhia Brasileira de Alumínio,Materials,Metals and Mining,Latin America and Caribbean,BR,BOVESPA,10.21,7.4000,4.500,-18.823127,9.525334,-32.021440,3.79,-1.976112
2227,DXCO3,Dexco S.A.,Materials,Paper and Forest Products,Latin America and Caribbean,BR,BOVESPA,5.88,7.3214,3.340,21.380013,11.688755,1.127385,96.09,1.829110
2228,TTKOM,Türk Telekomünikasyon Anonim Sirketi,Communication Services,Diversified Telecommunication Services,Africa / Middle East,TR,IBSE,68.15,83.5000,34.100,13.949832,10.579734,-5.117865,87.75,1.318543


In [3]:
%%sql
SELECT * FROM analytics.price_target_achievement WHERE upside_potential NOTNULL AND analyst_conviction NOTNULL

,ticker,name,country,exchange,sector,industry,achievement_probability,upside_potential,price_target_spread_pct,analyst_conviction,eps_revision_momentum,analyst_rating_normalized,expected_return_prob_weighted,confidence_level
0,QS,QuantumScape Corporation,US,NasdaqGS,Consumer Discretionary,Automobile Components,0.37,27.551020,120.000000,33.333333,0.000000,33.25,10.193878,Low
1,NWE,NorthWestern Energy Group Inc.,US,NasdaqGS,Utilities,Multi-Utilities,0.80,-6.501096,25.000000,14.285714,-0.001220,53.50,-5.200877,Medium
2,BIR,Birchcliff Energy Ltd.,CA,TSX,Energy,Oil Gas and Consumable Fuels,0.57,21.527778,22.857143,83.333333,-0.146705,83.25,12.270833,Medium
3,PRCH,Porch Group Inc.,US,NasdaqCM,Information Technology,Software,0.22,137.154150,50.000000,87.500000,0.000000,93.75,30.173913,Low
4,CXW,CoreCivic Inc.,US,NYSE,Industrials,Commercial Services and Supplies,0.35,60.810811,13.445378,75.000000,0.012735,100.00,21.283784,High
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2457,ADRO,PT Alamtri Resources Indonesia Tbk,ID,IDX,Energy,Oil Gas and Consumable Fuels,0.72,6.494595,80.000000,61.538462,-0.038005,77.00,4.676108,Low
2458,CBAV3,Companhia Brasileira de Alumínio,BR,BOVESPA,Materials,Metals and Mining,0.90,-27.522037,60.810811,87.500000,-0.083935,87.50,-24.769833,Low
2459,DXCO3,Dexco S.A.,BR,BOVESPA,Materials,Paper and Forest Products,0.50,24.513605,45.619690,80.000000,-0.242350,85.00,12.256803,Low
2460,TTKOM,Türk Telekomünikasyon Anonim Sirketi,TR,IBSE,Communication Services,Diversified Telecommunication Services,0.52,22.523844,40.838323,75.000000,0.407790,83.25,11.712399,Low


In [4]:
%%sql
SELECT * FROM analytics.kalman_filtered_price_targets

,ticker,name,country,exchange,sector,industry,kalman_estimate,kalman_variance,kalman_gain,signal_strength,original_price,original_target,filtered_upside
0,QS,QuantumScape Corporation,US,NasdaqGS,Consumer Discretionary,Automobile Components,11.029093,0.090909,0.909092,10.99999,8.82,11.2500,25.046405
1,NWE,NorthWestern Energy Group Inc.,US,NasdaqGS,Utilities,Multi-Utilities,64.404542,0.090909,0.909092,10.99999,68.45,64.0000,-5.910092
2,BIR,Birchcliff Energy Ltd.,CA,TSX,Energy,Oil Gas and Consumable Fuels,8.609092,0.090909,0.909092,10.99999,7.20,8.7500,19.570725
3,PRCH,Porch Group Inc.,US,NasdaqCM,Information Technology,Software,17.053645,0.090909,0.909092,10.99999,7.59,18.0000,124.685704
4,CXW,CoreCivic Inc.,US,NYSE,Industrials,Commercial Services and Supplies,28.727282,0.090909,0.909092,10.99999,18.50,29.7500,55.282606
...,...,...,...,...,...,...,...,...,...,...,...,...,...
2460,ADRO,PT Alamtri Resources Indonesia Tbk,ID,IDX,Energy,Oil Gas and Consumable Fuels,2351.072846,0.090909,0.909092,10.99999,2220.00,2364.1800,5.904182
2461,CBAV3,Companhia Brasileira de Alumínio,BR,BOVESPA,Materials,Metals and Mining,7.655452,0.090909,0.909092,10.99999,10.21,7.4000,-25.020057
2462,DXCO3,Dexco S.A.,BR,BOVESPA,Materials,Paper and Forest Products,7.190365,0.090909,0.909092,10.99999,5.88,7.3214,22.285116
2463,TTKOM,Türk Telekomünikasyon Anonim Sirketi,TR,IBSE,Communication Services,Diversified Telecommunication Services,82.104558,0.090909,0.909092,10.99999,68.15,83.5000,20.476241


In [5]:
%%sql
SELECT * FROM public.vw_features_analyst_sentiment

,isin,ticker,name,region,country,trading_country,exchange,sector,industry,dividend_record_frequency,...,pt_median_momentum_1m,pt_median_momentum_3m,pt_acceleration_short,pt_acceleration_long,pt_consensus_convergence,analyst_coverage_change_1m,analyst_coverage_change_3m,analyst_coverage_change_1y,pt_vs_price_momentum,analyst_coverage_trend
0,US67066G1040,NVDA,NVIDIA Corporation,United States and Canada,US,US,NasdaqGS,Information Technology,Semiconductors and Semiconductor Equipment,Quarterly,...,0.000000,0.111111,-0.091955,-0.375047,0.263111,1,2,4,0.113788,0.036207
1,US58507V1070,MDLN,Medline Inc.,United States and Canada,US,US,NasdaqGS,Health Care,Health Care Equipment and Supplies,NaN,...,NaN,NaN,NaN,NaN,NaN,26,26,26,NaN,1.000000
2,US0378331005,AAPL,Apple Inc.,United States and Canada,US,US,NasdaqGS,Information Technology,Technology Hardware Storage and Peripherals,Quarterly,...,0.000000,0.083032,-0.024503,-0.119223,-0.014019,0,0,1,0.041764,0.024390
3,US46222L1089,IONQ,IonQ Inc.,United States and Canada,US,US,NYSE,Information Technology,Technology Hardware Storage and Peripherals,NaN,...,0.000000,0.000000,0.003766,-0.692946,-0.112676,0,4,7,0.613063,0.223077
4,US02079K3059,GOOGL,Alphabet Inc.,United States and Canada,US,US,NasdaqGS,Communication Services,Interactive Media and Services,Quarterly,...,0.161388,0.170213,-0.062379,-0.530500,-0.138215,2,2,8,0.096603,0.040179
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
6405,NGCILEASING2,CILEASING,C & I Leasing Plc,Africa / Middle East,NG,NG,NGSE,Industrials,Trading Companies and Distributors,Annual,...,0.000000,0.000000,0.000000,NaN,0.000000,0,0,1,-0.358974,0.000000
6406,TN0006590012,SIAME,Société Industrielle d'Appareillage et de Maté...,Africa / Middle East,TN,TN,BVMT,Industrials,Electrical Equipment,Annual,...,0.049869,0.063830,-0.013961,0.223494,0.000000,0,0,0,-0.001082,0.000000
6407,NGBERGER0000,BERGER,Berger Paints Nigeria Plc,Africa / Middle East,NG,NG,NGSE,Materials,Chemicals,Interim Payment,...,0.000000,-0.027947,0.027947,0.000000,0.000000,0,0,0,-0.533414,0.000000
6408,TN0006530018,SOTET,Société Tunisienne d'Entreprises de Télécommun...,Africa / Middle East,TN,TN,BVMT,Communication Services,Diversified Telecommunication Services,Annual,...,0.000000,0.031206,-0.031206,0.094350,0.000000,0,0,0,-0.146987,0.000000


## 3. Data Overview & Quality Checks


In [6]:
# Rename the DataSpell-imported variables to convenient names
# (Adjust variable names if DataSpell assigns different ones)
try:
    mc = mc_sim.copy()
except NameError:
    print("⚠️ Run the data_input cells above first")

try:
    pt = pt_a.copy()
except NameError:
    print("⚠️ Run the price_target_achievement data_input cell first")

try:
    kal = pt_kal.copy()
except NameError:
    print("⚠️ Run the kalman_filtered_price_targets data_input cell first")

print(f"Monte Carlo Simulation:        {mc.shape[0]:,} stocks × {mc.shape[1]} cols")
print(f"Price Target Achievement:      {pt.shape[0]:,} stocks × {pt.shape[1]} cols")
print(f"Kalman Filtered Targets:       {kal.shape[0]:,} stocks × {kal.shape[1]} cols")

# Summary statistics for core return metrics
display(mc[["expected_upside_pct", "var_5_pct", "prob_positive_upside", "risk_reward_ratio"]].describe().round(2))


Monte Carlo Simulation:        2,230 stocks × 15 cols
Price Target Achievement:      2,462 stocks × 14 cols
Kalman Filtered Targets:       2,465 stocks × 13 cols


,expected_upside_pct,var_5_pct,prob_positive_upside,risk_reward_ratio
count,2230.00,2230.00,2230.00,2230.00
mean,24.53,4.54,75.32,2.87
std,40.22,30.00,32.55,13.02
min,-54.58,-64.97,0.00,-19.31
25%,1.14,-13.41,56.49,0.17
50%,14.45,-0.64,93.94,1.56
75%,35.06,15.96,100.00,3.20
max,597.30,264.50,100.00,530.33


## 4. Monte Carlo Simulation Analysis


### 4.1 Expected Upside Distribution


In [7]:
fig = make_subplots(
    rows=1, cols=2,
    subplot_titles=("Expected Upside Distribution", "Probability of Positive Return"),
    column_widths=[0.55, 0.45],
)

# Clip extreme outliers for better visualization
upside_clipped = mc["expected_upside_pct"].clip(-100, 300)

fig.add_trace(
    go.Histogram(
        x=upside_clipped,
        nbinsx=80,
        marker_color=COLORS[0],
        opacity=0.75,
        name="Expected Upside %",
    ),
    row=1, col=1,
)
fig.add_vline(x=0, line_dash="dash", line_color="red", row=1, col=1)
fig.add_vline(
    x=mc["expected_upside_pct"].median(),
    line_dash="dot", line_color="green",
    annotation_text=f"Median: {mc['expected_upside_pct'].median():.1f}%",
    row=1, col=1,
)

# Probability of positive return - pie chart
prob_bins = pd.cut(mc["prob_positive_upside"], bins=[0, 25, 50, 75, 100],
                   labels=["0-25%", "25-50%", "50-75%", "75-100%"])
prob_counts = prob_bins.value_counts().sort_index()
fig.add_trace(
    go.Bar(
        x=prob_counts.index.astype(str),
        y=prob_counts.values,
        marker_color=[COLORS[3], COLORS[1], COLORS[0], COLORS[2]],
        name="Stock Count",
    ),
    row=1, col=2,
)

fig.update_layout(
    title="Monte Carlo Simulation: Return Distribution Overview",
    template=PLOTLY_TEMPLATE,
    height=450,
    showlegend=True,
)
fig.update_xaxes(title_text="Expected Upside (%)", row=1, col=1)
fig.update_xaxes(title_text="Probability of Positive Return", row=1, col=2)
fig.update_yaxes(title_text="Number of Stocks", row=1, col=1)
fig.update_yaxes(title_text="Number of Stocks", row=1, col=2)
fig.show()


### 4.2 Risk-Reward by Industry


In [8]:
# Sector-level aggregation
mc_sector = (
    mc.groupby("industry")
    .agg(
        mean_upside=("expected_upside_pct", "mean"),
        median_upside=("expected_upside_pct", "median"),
        mean_var5=("var_5_pct", "mean"),
        mean_prob_positive=("prob_positive_upside", "mean"),
        count=("ticker", "count"),
    )
    .reset_index()
    .sort_values("mean_upside", ascending=False)
)

fig = px.scatter(
    mc_sector,
    x="mean_var5",
    y="mean_upside",
    size="count",
    color="industry",
    hover_name="industry",
    hover_data={"mean_prob_positive": ":.1f", "count": True},
    title="Industry Risk-Reward: Expected Upside vs Value-at-Risk (5%)",
    labels={
        "mean_var5": "Mean VaR 5% (%)",
        "mean_upside": "Mean Expected Upside (%)",
        "count": "# Stocks",
    },
    template=PLOTLY_TEMPLATE,
    height=500,
)
fig.add_hline(y=0, line_dash="dash", line_color="gray", opacity=0.5)
fig.add_vline(x=0, line_dash="dash", line_color="gray", opacity=0.5)
fig.show()


### 4.3 Top Opportunities — Highest Risk-Reward Ratio (Positive Upside)


In [9]:
mc_positive = mc[mc["prob_positive_upside"] >= 75].nlargest(50, "risk_reward_ratio")

fig = px.bar(
    mc_positive,
    x="ticker",
    y="expected_upside_pct",
    color="industry",
    hover_data=["name", "prob_positive_upside", "risk_reward_ratio"],
    title="Top 50 Opportunities: Highest Risk-Reward (≥75% Prob Positive)",
    labels={"expected_upside_pct": "Expected Upside (%)", "ticker": "Ticker"},
    template=PLOTLY_TEMPLATE,
    height=500,
)
fig.update_layout(xaxis_tickangle=-45)
fig.show()


## 5. Price Target Achievement Analysis


### 5.1 Achievement Probability Distribution by Confidence Level


In [10]:
fig = px.violin(
    pt,
    x="confidence_level",
    y="achievement_probability",
    color="confidence_level",
    box=True,
    points="outliers",
    title="Price Target Achievement Probability by Confidence Level",
    labels={
        "achievement_probability": "Achievement Probability",
        "confidence_level": "Confidence Level",
    },
    category_orders={"confidence_level": ["Low", "Medium", "High"]},
    color_discrete_sequence=[COLORS[3], COLORS[1], COLORS[2]],
    template=PLOTLY_TEMPLATE,
    height=450,
)
fig.show()


### 5.2 Probability-Weighted Expected Return by Sector


In [11]:
pt_sector = (
    pt.groupby("industry")
    .agg(
        mean_expected_return=("expected_return_prob_weighted", "mean"),
        median_expected_return=("expected_return_prob_weighted", "median"),
        mean_achievement_prob=("achievement_probability", "mean"),
        mean_conviction=("analyst_conviction", "mean"),
        count=("ticker", "count"),
    )
    .reset_index()
    .sort_values("mean_expected_return", ascending=True)
)

fig = go.Figure()
fig.add_trace(
    go.Bar(
        y=pt_sector["industry"],
        x=pt_sector["mean_expected_return"],
        orientation="h",
        marker_color=[
            COLORS[2] if v >= 0 else COLORS[3]
            for v in pt_sector["mean_expected_return"]
        ],
        text=pt_sector["mean_expected_return"].apply(lambda v: f"{v:.1f}%"),
        textposition="outside",
        hovertemplate=(
            "<b>%{y}</b><br>"
            "Mean Prob-Weighted Return: %{x:.2f}%<br>"
            "Avg Achievement Prob: %{customdata[0]:.0%}<br>"
            "Avg Conviction: %{customdata[1]:.1f}<br>"
            "Stocks: %{customdata[2]}"
        ),
        customdata=pt_sector[["mean_achievement_prob", "mean_conviction", "count"]].values,
    )
)
fig.update_layout(
    title="Probability-Weighted Expected Return by Industry",
    xaxis_title="Mean Expected Return (%)",
    template=PLOTLY_TEMPLATE,
    height=1100,
    margin=dict(l=350),
)
fig.show()


### 5.3 Conviction vs Upside Potential Scatter


In [12]:
fig = px.scatter(
    pt.sample(min(2000, len(pt)), random_state=42),
    x="expected_return_prob_weighted",
    y="upside_potential",
    color="achievement_probability",
    size="analyst_conviction",
    hover_name="ticker",
    hover_data=["name", "sector", "industry", "expected_return_prob_weighted", "confidence_level"],
    title="Analyst Conviction vs Upside Potential",
    labels={
        "analyst_conviction": "Analyst Conviction (%)",
        "upside_potential": "Upside Potential (%)",
    },
    category_orders={"confidence_level": ["Low", "Medium", "High"]},
    color_discrete_sequence=[COLORS[3], COLORS[1], COLORS[2]],
    template=PLOTLY_TEMPLATE,
    height=500,
    opacity=0.6,
)
fig.add_hline(y=0, line_dash="dash", line_color="gray", opacity=0.4)
fig.show()


## 6. Kalman Filtered Price Target Analysis


### 6.1 Kalman Filtered vs Original Upside


In [13]:
# Pre-compute the column on the full DataFrame
kal["raw_upside"] = (kal["original_target"] - kal["original_price"]) / kal["original_price"] * 100

# Sample AFTER the column exists
kal_sample = kal.sample(min(2000, len(kal)), random_state=42).copy()

# Apply signed log1p transform for axis-aligned visualization
kal_sample["filtered_upside_log"] = np.sign(kal_sample["filtered_upside"]) * np.log1p(
    np.abs(kal_sample["filtered_upside"]))
kal_sample["raw_upside_log"] = np.sign(kal_sample["raw_upside"]) * np.log1p(np.abs(kal_sample["raw_upside"]))

fig = px.scatter(
    kal_sample,
    x="filtered_upside_log",
    y="raw_upside_log",
    color="industry",
    hover_name="ticker",
    hover_data=["name", "kalman_estimate", "original_target", "original_price", "filtered_upside", "raw_upside"],
    title="Kalman-Filtered Upside vs Raw Analyst Upside (Log-Transformed Axes)",
    labels={
        "filtered_upside_log": "Kalman Filtered Upside — sign(x)·log₁ₚ(|x|)",
        "raw_upside_log": "Raw Analyst Upside — sign(x)·log₁ₚ(|x|)",
    },
    template=PLOTLY_TEMPLATE,
    height=500,
    opacity=0.6,
)

# Add diagonal reference line on the log-transformed scale
log_max = max(
    kal_sample["filtered_upside_log"].abs().quantile(0.99),
    kal_sample["raw_upside_log"].abs().quantile(0.99),
)
fig.add_shape(
    type="line", x0=-log_max, y0=-log_max, x1=log_max, y1=log_max,
    line=dict(color="gray", dash="dash", width=1),
)
fig.show()


### 6.2 Signal Strength Distribution by Sector


In [14]:
fig = px.box(
    kal,
    x="industry",
    y="filtered_upside",
    color="industry",
    title="Kalman-Filtered Upside Distribution by Sector",
    labels={
        "filtered_upside": "Filtered Upside (%)",
        "industry": "",
    },
    template=PLOTLY_TEMPLATE,
    height=1000,
)
fig.update_layout(
    xaxis_tickangle=-65,
    showlegend=False,
)
fig.add_hline(y=0, line_dash="dash", line_color="red", opacity=0.5)
fig.show()


### 6.3 Kalman Noise Reduction Effectiveness


In [15]:
kal["raw_upside"] = (kal["original_target"] - kal["original_price"]) / kal["original_price"] * 100
kal["noise_reduction"] = abs(kal["raw_upside"] - kal["filtered_upside"])

noise_by_sector = (
    kal.groupby("industry")
    .agg(
        mean_noise_reduction=("noise_reduction", "mean"),
        median_raw_upside=("raw_upside", "median"),
        median_filtered_upside=("filtered_upside", "median"),
        count=("ticker", "count"),
    )
    .reset_index()
    .sort_values("mean_noise_reduction", ascending=False)
)

fig = go.Figure()
fig.add_trace(go.Bar(
    x=noise_by_sector["industry"],
    y=noise_by_sector["median_raw_upside"],
    name="Raw Median Upside",
    marker_color=COLORS[1],
    opacity=0.7,
))
fig.add_trace(go.Bar(
    x=noise_by_sector["industry"],
    y=noise_by_sector["median_filtered_upside"],
    name="Kalman-Filtered Median Upside",
    marker_color=COLORS[0],
))
fig.update_layout(
    title="Kalman Filter Impact: Raw vs Filtered Median Upside by Sector",
    yaxis_title="Median Upside (%)",
    barmode="group",
    template=PLOTLY_TEMPLATE,
    height=1000,
    xaxis_tickangle=-85,
)
fig.show()


## 7. Cross-Model Comparison


### 7.1 MC Expected Upside vs Kalman Filtered Upside


In [16]:
# Merge Monte Carlo and Kalman results
mc_kal = mc.merge(
    kal[["ticker", "country", "exchange", "filtered_upside", "kalman_estimate", "original_price", "original_target"]],
    on="ticker",
    how="inner",
)

fig = px.scatter(
    mc_kal.sample(min(2000, len(mc_kal)), random_state=42),
    x="expected_upside_pct",
    y="filtered_upside",
    color="industry",
    hover_name="ticker",
    hover_data=["name", "original_price", "kalman_estimate", "original_target", "prob_positive_upside"],
    title="Monte Carlo vs Kalman-Filtered Expected Returns",
    labels={
        "expected_upside_pct": "MC Expected Upside (%)",
        "filtered_upside": "Kalman Filtered Upside (%)",
    },
    template=PLOTLY_TEMPLATE,
    height=500,
    opacity=0.5,
)
# Diagonal reference
fig.add_shape(
    type="line", x0=-100, y0=-100, x1=200, y1=200,
    line=dict(color="red", dash="dash", width=2),
)
fig.show()

In [17]:

# Correlation summary
corr = mc_kal[["expected_upside_pct", "filtered_upside"]].corr().iloc[0, 1]
print(f"📊 MC ↔ Kalman correlation: {corr:.3f}")


📊 MC ↔ Kalman correlation: 0.931


### 7.2 Tri-Model Alignment: MC + Kalman + Achievement


In [18]:
# Merge all three models
tri = (
    mc[["ticker", "name", "sector", "industry", "expected_upside_pct", "prob_positive_upside"]]
    .merge(
        kal[["ticker", "filtered_upside"]],
        on="ticker",
        how="inner",
    )
    .merge(
        pt[["ticker", "expected_return_prob_weighted", "achievement_probability", "confidence_level"]],
        on="ticker",
        how="inner",
    )
)

# Agreement score: all three models agree on direction
tri["mc_bullish"] = tri["expected_upside_pct"] > 0
tri["kal_bullish"] = tri["filtered_upside"] > 0
tri["pt_bullish"] = tri["expected_return_prob_weighted"] > 0
tri["agreement_score"] = (
        tri["mc_bullish"].astype(int)
        + tri["kal_bullish"].astype(int)
        + tri["pt_bullish"].astype(int)
)
tri["signal"] = tri["agreement_score"].map(
    {0: "Strong Bearish (0/3)", 1: "Bearish (1/3)", 2: "Bullish (2/3)", 3: "Strong Bullish (3/3)"}
)

fig = px.histogram(
    tri,
    x="signal",
    color="signal",
    title="Tri-Model Signal Agreement (MC + Kalman + Achievement)",
    labels={"signal": "Model Agreement", "count": "Number of Stocks"},
    color_discrete_map={
        "Strong Bearish (0/3)": COLORS[3],
        "Bearish (1/3)": COLORS[1],
        "Bullish (2/3)": COLORS[0],
        "Strong Bullish (3/3)": COLORS[2],
    },
    category_orders={"signal": [
        "Strong Bearish (0/3)", "Bearish (1/3)",
        "Bullish (2/3)", "Strong Bullish (3/3)",
    ]},
    template=PLOTLY_TEMPLATE,
    height=420,
)
fig.update_layout(showlegend=False)
fig.show()

In [19]:

print(f"\n📊 Model Agreement Summary:")
print(tri["signal"].value_counts().to_string())



📊 Model Agreement Summary:
signal
Strong Bullish (3/3)    1644
Strong Bearish (0/3)     404
Bullish (2/3)            104
Bearish (1/3)             78


### 7.3 Strong Consensus Picks — All 3 Models Bullish, High Confidence


In [20]:
strong_consensus = (
    tri[
        (tri["agreement_score"] == 3)
        & (tri["prob_positive_upside"] >= 55)
        & (tri["achievement_probability"] >= 0.6)
        ]
    .nlargest(50, "expected_upside_pct")
)

if len(strong_consensus) > 0:
    fig = go.Figure()
    fig.add_trace(go.Bar(
        x=strong_consensus["ticker"],
        y=strong_consensus["expected_upside_pct"],
        name="MC Expected Upside",
        marker_color=COLORS[0],
    ))
    fig.add_trace(go.Bar(
        x=strong_consensus["ticker"],
        y=strong_consensus["filtered_upside"],
        name="Kalman Filtered Upside",
        marker_color=COLORS[1],
    ))
    fig.add_trace(go.Bar(
        x=strong_consensus["ticker"],
        y=strong_consensus["expected_return_prob_weighted"],
        name="Prob-Weighted Return",
        marker_color=COLORS[2],
    ))
    fig.update_layout(
        title=f"Top {len(strong_consensus)} Strong Consensus Picks (All 3 Models Bullish)",
        yaxis_title="Expected Return (%)",
        barmode="group",
        template=PLOTLY_TEMPLATE,
        height=500,
        xaxis_tickangle=-45,
    )
    fig.show()

    display(
        strong_consensus[["ticker", "name", "sector", "industry", "expected_upside_pct",
                          "filtered_upside", "expected_return_prob_weighted",
                          "prob_positive_upside", "achievement_probability", "confidence_level"]]
        .reset_index(drop=True)
    )
else:
    print("No stocks meet the strong consensus criteria.")


,ticker,name,sector,industry,expected_upside_pct,filtered_upside,expected_return_prob_weighted,prob_positive_upside,achievement_probability,confidence_level
0,TLEVISACPO,Grupo Televisa S.A.B.,Communication Services,Diversified Telecommunication Services,124.085786,5.121644,3.774648,97.56,0.67,Low
1,ARA,Consorcio ARA S. A. B. de C. V.,Consumer Discretionary,Household Durables,80.410207,3.932458,2.681934,99.37,0.62,Low
2,ORBIA,Orbia Advance Corporation S.A.B. de C.V.,Materials,Chemicals,47.493697,6.439436,4.745860,84.36,0.67,Low
3,ANAB,AnaptysBio Inc.,Health Care,Biotechnology,45.141620,16.865503,11.502262,87.90,0.62,Low
4,WAVE,Wavestone SA,Information Technology,IT Services,40.928033,42.596388,28.582150,100.00,0.61,High
5,LNTH,Lantheus Holdings Inc.,Health Care,Health Care Equipment and Supplies,37.156474,13.133727,9.246136,100.00,0.64,Low
6,BKSY,BlackSky Technology Inc.,Industrials,Professional Services,36.294809,14.548911,9.602273,98.74,0.60,Low
7,IRB,IRB Infrastructure Developers Limited,Industrials,Construction and Engineering,35.854639,25.719746,17.823768,100.00,0.63,Low
8,BRNL,Brunel International N.V.,Industrials,Professional Services,33.663248,17.925753,12.619718,100.00,0.64,Low
9,NXR,Norcros plc,Industrials,Building Products,32.354075,13.496320,9.501401,100.00,0.64,Low


## 8. Sector Expected Returns Heatmap


In [21]:
# Aggregate all return metrics by sector
sector_returns = (
    tri.groupby("industry")
    .agg(
        mc_mean=("expected_upside_pct", "mean"),
        mc_median=("expected_upside_pct", "median"),
        kalman_mean=("filtered_upside", "mean"),
        kalman_median=("filtered_upside", "median"),
        pt_mean=("expected_return_prob_weighted", "mean"),
        pt_median=("expected_return_prob_weighted", "median"),
        pct_bullish=("agreement_score", lambda x: (x == 3).mean() * 100),
        count=("ticker", "count"),
    )
    .reset_index()
)

heatmap_data = sector_returns.set_index("industry")[
    ["mc_mean", "mc_median", "kalman_mean", "kalman_median", "pt_mean", "pt_median", "pct_bullish"]
].rename(columns={
    "mc_mean": "MC Mean",
    "mc_median": "MC Median",
    "kalman_mean": "Kalman Mean",
    "kalman_median": "Kalman Median",
    "pt_mean": "Achiev. Mean",
    "pt_median": "Achiev. Median",
    "pct_bullish": "% All Bullish",
})

fig = px.imshow(
    heatmap_data.round(1),
    color_continuous_scale="RdYlGn",
    text_auto=True,
    aspect="auto",
    title="Sector Expected Returns Heatmap (All Models)",
    labels={"color": "Value"},
)
fig.update_layout(
    template=PLOTLY_TEMPLATE,
    height=1250,
)
fig.show()


## 9. VaR & Tail Risk Analysis


In [22]:
fig = make_subplots(
    rows=1, cols=2,
    subplot_titles=("VaR 5% Distribution", "VaR 5% vs Expected Upside"),
)

# VaR distribution
var_clipped = mc["var_5_pct"].clip(-150, 300)
fig.add_trace(
    go.Histogram(
        x=var_clipped,
        nbinsx=80,
        marker_color=COLORS[3],
        opacity=0.75,
        name="VaR 5%",
    ),
    row=1, col=1,
)
fig.add_vline(x=0, line_dash="dash", line_color="blue", row=1, col=1)

# VaR vs Expected Upside (sampled for performance)
sample = mc.sample(min(2000, len(mc)), random_state=42)
fig.add_trace(
    go.Scatter(
        x=sample["var_5_pct"],
        y=sample["expected_upside_pct"],
        mode="markers",
        marker=dict(
            size=4,
            color=sample["prob_positive_upside"],
            colorscale="RdYlGn",
            colorbar=dict(title="P(+)"),
            opacity=0.5,
        ),
        name="Stocks",
    ),
    row=1, col=2,
)
fig.add_shape(
    type="line", x0=-100, y0=-100, x1=300, y1=300,
    line=dict(color="gray", dash="dash", width=1),
    row=1, col=2,
)

fig.update_layout(
    title="Value-at-Risk (5%) Analysis",
    template=PLOTLY_TEMPLATE,
    height=450,
    showlegend=False,
)
fig.update_xaxes(title_text="VaR 5% (%)", row=1, col=1)
fig.update_xaxes(title_text="VaR 5% (%)", row=1, col=2)
fig.update_yaxes(title_text="Count", row=1, col=1)
fig.update_yaxes(title_text="Expected Upside (%)", row=1, col=2)
fig.show()


## 9.5 InferenceData Schema Integration

Build ArviZ-compatible InferenceData from Monte Carlo simulation results
for standardised posterior analysis, diagnostics, and NetCDF export.


In [23]:
# Build InferenceData from Monte Carlo simulation results
if ARVIZ_AVAILABLE and 'mc' in dir() and len(mc) > 0:
    try:
        idata_mc = build_monte_carlo_inference_data(
            mc, mc, n_simulations=10000,
        )
        mc_summary = summarize_inference_data(idata_mc)
        print(f"✅ InferenceData built: {mc_summary.get('groups', [])}")
        print(f"   Draws: {mc_summary.get('n_draws', 0)}, Equities: {mc_summary.get('n_equities', 0)}")
        if mc_summary.get('r_hat'):
            for var, rhat_val in mc_summary['r_hat'].items():
                print(f"   R-hat ({var}): {rhat_val:.4f}")
    except Exception as e:
        print(f"⚠️ InferenceData build failed: {e}")
else:
    print('⚠️ ArviZ not available or no MC data')


✅ InferenceData built: ['posterior_predictive', 'observed_data', 'constant_data']
   Draws: 10000, Equities: 2230


## 10. Summary Statistics


In [24]:
summary = {
    "Monte Carlo": {
        "Stocks Analyzed": len(mc),
        "Mean Expected Upside (%)": mc["expected_upside_pct"].mean().round(2),
        "Median Expected Upside (%)": mc["expected_upside_pct"].median().round(2),
        "% Stocks with Positive Upside": (mc["expected_upside_pct"] > 0).mean() * 100,
        "Mean Prob Positive (%)": mc["prob_positive_upside"].mean().round(1),
    },
    "Price Target Achievement": {
        "Stocks Analyzed": len(pt),
        "Mean Prob-Weighted Return (%)": pt["expected_return_prob_weighted"].mean().round(2),
        "Mean Achievement Prob": pt["achievement_probability"].mean().round(3),
        "High Confidence Count": (pt["confidence_level"] == "High").sum(),
        "Mean Analyst Conviction (%)": pt["analyst_conviction"].mean().round(1),
    },
    "Kalman Filter": {
        "Stocks Analyzed": len(kal),
        "Mean Filtered Upside (%)": kal["filtered_upside"].mean().round(2),
        "Median Filtered Upside (%)": kal["filtered_upside"].median().round(2),
        "Mean Signal Strength": kal["signal_strength"].mean().round(2),
        "% Positive Filtered Upside": (kal["filtered_upside"] > 0).mean() * 100,
    },
}

summary_df = pd.DataFrame(summary).T
display(summary_df)

if len(tri) > 0:
    print(f"\n🔗 Cross-Model Coverage: {len(tri):,} stocks in all 3 models")
    print(
        f"   Strong Bullish (3/3 agree): {(tri['agreement_score'] == 3).sum():,} ({(tri['agreement_score'] == 3).mean() * 100:.1f}%)")
    print(
        f"   Strong Bearish (0/3 agree): {(tri['agreement_score'] == 0).sum():,} ({(tri['agreement_score'] == 0).mean() * 100:.1f}%)")
    print(f"   MC ↔ Kalman correlation:    {tri[['expected_upside_pct', 'filtered_upside']].corr().iloc[0, 1]:.3f}")
    print(
        f"   MC ↔ Achievement corr:      {tri[['expected_upside_pct', 'expected_return_prob_weighted']].corr().iloc[0, 1]:.3f}")

print("\n✅ Expected Returns Analytics complete")


,Stocks Analyzed,Mean Expected Upside (%),Median Expected Upside (%),% Stocks with Positive Upside,Mean Prob Positive (%),Mean Prob-Weighted Return (%),Mean Achievement Prob,High Confidence Count,Mean Analyst Conviction (%),Mean Filtered Upside (%),Median Filtered Upside (%),Mean Signal Strength,% Positive Filtered Upside
Monte Carlo,2230.0,24.53,14.45,77.219731,75.3,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Price Target Achievement,2462.0,NaN,NaN,NaN,NaN,7.31,0.594,392.0,58.8,NaN,NaN,NaN,NaN
Kalman Filter,2465.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,21.93,12.73,11.0,78.377282



🔗 Cross-Model Coverage: 2,230 stocks in all 3 models
   Strong Bullish (3/3 agree): 1,644 (73.7%)
   Strong Bearish (0/3 agree): 404 (18.1%)
   MC ↔ Kalman correlation:    0.931
   MC ↔ Achievement corr:      0.750

✅ Expected Returns Analytics complete
